In [1]:
!pip install sentence-transformers faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 28.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

import google.generativeai as genai

from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
GOOGLE_API_KEY = userdata.get(
    "Research++"
)

genai.configure(
    api_key=GOOGLE_API_KEY
)

llm = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print(
    "Gemini configured successfully."
)

Gemini configured successfully.


In [5]:
from google.colab import files

uploaded = files.upload()
uploaded = files.upload()

Saving papers.json to papers (1).json


Saving researchforge_faiss.index to researchforge_faiss (1).index


In [6]:
df = pd.read_json(
    "papers.json"
)

faiss_index = faiss.read_index(
    "researchforge_faiss.index"
)

print(
    f"Loaded {len(df)} papers."
)

Loaded 100 papers.


In [7]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print(
    "Embedding model ready."
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.


In [8]:
def retrieve_papers(
    user_query,
    top_k=10
):

    query_embedding = embedding_model.encode(
        [user_query]
    )

    distances, indices = faiss_index.search(
        np.array(
            query_embedding
        ).astype("float32"),
        k=top_k
    )

    retrieved_papers = []

    for idx in indices[0]:

        retrieved_papers.append({

            "title":
            df.iloc[idx]["title"],

            "abstract":
            df.iloc[idx]["abstract"],

            "published":
            df.iloc[idx]["published"]

        })

    return retrieved_papers

In [9]:
def detect_research_gaps(
    user_query,
    retrieved_papers
):

    evidence_context = ""

    for paper in retrieved_papers:

        evidence_context += f"""

Title:
{paper['title']}

Abstract:
{paper['abstract']}

"""

    prompt = f"""

You are an expert AI research strategist.

User Question:

{user_query}

Retrieved Scientific Evidence:

{evidence_context}

Task:

Identify research gaps and
future opportunities.

Return:

1. Underexplored problems

2. Weaknesses in current methods

3. Dataset limitations

4. Evaluation limitations

5. Open scientific challenges

6. Potential future directions

7. Novel project / research ideas

Only use retrieved evidence.

Do not invent unsupported claims.

"""

    response = llm.generate_content(
        prompt
    )

    return response.text

In [10]:
query = """
Find research gaps in
multimodal deepfake detection.
"""

papers = retrieve_papers(
    query
)

gap_analysis = detect_research_gaps(
    query,
    papers
)

print(
    gap_analysis
)

Here are the research gaps and future opportunities in multimodal deepfake detection, derived solely from the provided scientific evidence:

1.  **Underexplored problems**
    *   Detecting dynamic deepfake attacks with challenging frame-by-frame alterations, which current benchmarks struggle with. (WWW)
    *   Identifying manipulated segments within both video and audio (clip-level evaluation) to provide insight into the origins of deepfakes. (WWW)
    *   Addressing deepfakes involving multilingual audio content, as most existing multimodal datasets are limited to a single language. (PolyGlotFake)
    *   Comprehensive analysis and deeper exploration of findings related to multiple perspectives, such as augmentation and stacked forgery, in multimodal deepfake detection. (DeepfakeBench-MM)

2.  **Weaknesses in current methods**
    *   Most current multimodal detectors are small, task-specific, scale poorly, and generalize weakly across domains. (AV-LMMDetect)
    *   Existing multim